In [1]:
#Импорты и загрузка .env
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# Находим корень проекта при запуске из него или из notebooks.
current_dir = Path.cwd().resolve()
project_dir = next(
    (
        directory
        for directory in [current_dir, *current_dir.parents]
        if (directory / ".env").is_file()
    ),
    None,
)

if project_dir is None:
    raise FileNotFoundError("Не найден .env в каталоге проекта.")

load_dotenv(project_dir / ".env")

print("Каталог проекта:", project_dir)

Каталог проекта: /home/mle-user/mle_project/mle-project_Mandrygina_Olga_M25-555


In [2]:
#Подключение и чтение датасета
required = [
    "DB_DESTINATION_HOST",
    "DB_DESTINATION_PORT",
    "DB_DESTINATION_NAME",
    "DB_DESTINATION_USER",
    "DB_DESTINATION_PASSWORD",
]

missing = [name for name in required if not os.getenv(name)]

if missing:
    raise ValueError("Не заполнены переменные: " + ", ".join(missing))

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["DB_DESTINATION_USER"],
    password=os.environ["DB_DESTINATION_PASSWORD"],
    host=os.environ["DB_DESTINATION_HOST"],
    port=int(os.environ["DB_DESTINATION_PORT"]),
    database=os.environ["DB_DESTINATION_NAME"],
)

engine = create_engine(
    url,
    connect_args={"connect_timeout": 10},
)

try:
    with engine.connect() as conn:
        df = pd.read_sql_query(
            text("""
                SELECT *
                FROM public.flats_dataset
                ORDER BY flat_id
            """),
            conn,
        )
finally:
    engine.dispose()

print(f"Строк: {df.shape[0]:,}")
print(f"Колонок: {df.shape[1]}")
display(df.head())

Строк: 141,362
Колонок: 18


,flat_id,building_id,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,False,9.900,19.900,1,False,35.100,9500000,1965,6,55.717,37.781,2.640,84,12,True
1,1,18012,7,False,0.000,16.600,1,False,43.000,13500000,2001,2,55.795,37.608,3.000,97,10,True
2,2,17821,9,False,9.000,32.000,2,False,56.000,13500000,2000,4,55.740,37.762,2.700,80,10,True
3,3,18579,1,False,10.100,43.100,3,False,76.000,20000000,2002,4,55.672,37.571,2.640,771,17,True
4,4,9293,3,False,3.000,14.000,1,False,24.000,5200000,1971,1,55.809,37.707,2.600,208,9,True


In [3]:
#Типы данных и пропуски
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141362 entries, 0 to 141361
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flat_id            141362 non-null  int64  
 1   building_id        141362 non-null  int64  
 2   floor              141362 non-null  int64  
 3   is_apartment       141362 non-null  bool   
 4   kitchen_area       141362 non-null  float64
 5   living_area        141362 non-null  float64
 6   rooms              141362 non-null  int64  
 7   studio             141362 non-null  bool   
 8   total_area         141362 non-null  float64
 9   price              141362 non-null  int64  
 10  build_year         141362 non-null  int64  
 11  building_type_int  141362 non-null  int64  
 12  latitude           141362 non-null  float64
 13  longitude          141362 non-null  float64
 14  ceiling_height     141362 non-null  float64
 15  flats_count        141362 non-null  int64  
 16  fl

In [4]:
#Типы данных и пропуски
quality_report = pd.DataFrame({
    "Тип данных": df.dtypes.astype(str),
    "Пропусков": df.isna().sum(),
    "Пропусков, %": df.isna().mean().mul(100).round(2),
    "Уникальных значений": df.nunique(dropna=True),
}).sort_values("Пропусков", ascending=False)

display(quality_report)

,Тип данных,Пропусков,"Пропусков, %",Уникальных значений
flat_id,int64,0,0.000,141362
building_id,int64,0,0.000,24620
floor,int64,0,0.000,56
is_apartment,bool,0,0.000,2
kitchen_area,float64,0,0.000,1036
living_area,float64,0,0.000,2345
rooms,int64,0,0.000,14
studio,bool,0,0.000,1
total_area,float64,0,0.000,3358
price,int64,0,0.000,8384


In [5]:
#Проверка дубликатов
comparison_columns = [
    column for column in df.columns
    if column != "flat_id"
]

duplicate_report = pd.Series({
    "Полные дубликаты строк": int(df.duplicated().sum()),
    "Повторные flat_id": int(df.duplicated(subset=["flat_id"]).sum()),
    "Повторы всех полей кроме flat_id": int(
        df.duplicated(subset=comparison_columns).sum()
    ),
})

display(duplicate_report)

Полные дубликаты строк                 0
Повторные flat_id                      0
Повторы всех полей кроме flat_id    9116
dtype: int64

In [6]:
#Проверка дубликатов
duplicate_mask = df.duplicated(
    subset=comparison_columns,
    keep=False,
)

display(
    df.loc[duplicate_mask]
    .sort_values(["building_id", "floor", "total_area", "price"])
    .head(20)
)

,flat_id,building_id,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
52640,52640,3,3,False,0.000,0.000,4,False,120.000,60000000,1901,1,55.760,37.646,3.200,26,4,False
54009,54009,3,3,False,0.000,0.000,4,False,120.000,60000000,1901,1,55.760,37.646,3.200,26,4,False
90882,90882,4,4,False,11.000,48.000,2,False,63.000,28000000,1902,1,55.749,37.656,3.000,32,4,False
122183,122183,4,4,False,11.000,48.000,2,False,63.000,28000000,1902,1,55.749,37.656,3.000,32,4,False
63568,63568,6,5,False,10.000,74.000,3,False,100.000,50000000,1902,1,55.755,37.595,3.200,15,6,True
63594,63594,6,5,False,10.000,74.000,3,False,100.000,50000000,1902,1,55.755,37.595,3.200,15,6,True
14477,14477,10,3,False,0.000,0.000,3,False,85.000,55500000,1902,1,55.764,37.603,3.100,20,22,False
107057,107057,10,3,False,0.000,0.000,3,False,85.000,55500000,1902,1,55.764,37.603,3.100,20,22,False
115310,115310,10,3,False,0.000,0.000,3,False,85.000,55500000,1902,1,55.764,37.603,3.100,20,22,False
138158,138158,13,8,False,15.200,90.200,4,False,130.000,49900000,1902,1,55.768,37.599,3.000,40,9,True


In [7]:
#Распределения числовых значений
#Идентификаторы и код типа здания исключим из числовой статистики

numeric_columns = [
    "price",
    "total_area",
    "living_area",
    "kitchen_area",
    "rooms",
    "floor",
    "floors_total",
    "ceiling_height",
    "build_year",
    "flats_count",
    "latitude",
    "longitude",
]

numeric_summary = df[numeric_columns].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T

display(numeric_summary)

,count,mean,std,min,1%,5%,50%,95%,99%,max
price,"141,362.000","19,441,619.952","66,269,544.299",11.000,"5,200,000.000","6,500,000.000","11,850,000.000","49,500,000.000","152,000,000.000","9,873,737,728.000"
total_area,"141,362.000",62.375,40.296,11.000,29.000,32.500,53.000,127.000,223.900,960.300
living_area,"141,362.000",31.057,23.969,0.000,0.000,0.000,29.400,67.000,119.600,700.000
kitchen_area,"141,362.000",9.002,5.264,0.000,0.000,0.000,8.800,17.300,28.000,203.000
rooms,"141,362.000",2.129,0.994,1.000,1.000,1.000,2.000,4.000,5.000,20.000
floor,"141,362.000",7.467,5.717,1.000,1.000,1.000,6.000,18.000,25.000,56.000
floors_total,"141,362.000",14.108,6.898,1.000,4.000,5.000,14.000,25.000,40.000,99.000
ceiling_height,"141,362.000",2.754,0.223,2.000,2.480,2.480,2.640,3.100,3.400,27.000
build_year,"141,362.000","1,986.600",22.136,"1,901.000","1,926.000","1,957.000","1,985.000","2,017.000","2,019.000","2,023.000"
flats_count,"141,362.000",251.993,207.336,1.000,19.000,58.000,200.000,624.000,"1,056.000","4,455.000"


In [8]:
#Для категориальных и логических признаков
for column in [
    "building_type_int",
    "is_apartment",
    "studio",
    "has_elevator",
]:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False))


building_type_int


building_type_int
4    78696
2    25265
1    23164
6    10533
0     1927
3     1773
5        4
Name: count, dtype: int64


is_apartment


is_apartment
False    139990
True       1372
Name: count, dtype: int64


studio


studio
False    141362
Name: count, dtype: int64


has_elevator


has_elevator
True     126856
False     14506
Name: count, dtype: int64

In [9]:
#Проверка подозрительных значений
#Здесь считаем нарушения и случаи для изучения.
#Не каждое условие означает, что строку нужно удалить:
# например, ноль комнат может обозначать студию,
# а будущий год постройки — новостройку.

checks = {
    "Цена <= 0": df["price"] <= 0,
    "Общая площадь <= 0": df["total_area"] <= 0,
    "Жилая площадь < 0": df["living_area"] < 0,
    "Площадь кухни < 0": df["kitchen_area"] < 0,
    "Жилая площадь > общей": df["living_area"] > df["total_area"],
    "Площадь кухни > общей": df["kitchen_area"] > df["total_area"],
    "Жилая + кухня > общей": (
        df["living_area"] + df["kitchen_area"] > df["total_area"]
    ),
    "Количество комнат < 0": df["rooms"] < 0,
    "Количество комнат = 0": df["rooms"] == 0,
    "Этаж <= 0": df["floor"] <= 0,
    "Этажность дома <= 0": df["floors_total"] <= 0,
    "Этаж квартиры > этажности дома": df["floor"] > df["floors_total"],
    "Высота потолков <= 0": df["ceiling_height"] <= 0,
    "Год постройки <= 0": df["build_year"] <= 0,
    "Количество квартир в доме <= 0": df["flats_count"] <= 0,
    "Широта за пределами [-90, 90]": (
        df["latitude"].notna()
        & ~df["latitude"].between(-90, 90)
    ),
    "Долгота за пределами [-180, 180]": (
        df["longitude"].notna()
        & ~df["longitude"].between(-180, 180)
    ),
}

anomaly_report = pd.DataFrame([
    {
        "Проверка": name,
        "Количество": int(mask.fillna(False).sum()),
        "Доля, %": round(mask.fillna(False).mean() * 100, 2),
    }
    for name, mask in checks.items()
]).sort_values("Количество", ascending=False)

display(anomaly_report)

print("Бесконечные значения по колонкам:")
display(np.isinf(df[numeric_columns]).sum())

,Проверка,Количество,"Доля, %"
6,Жилая + кухня > общей,152,0.110
5,Площадь кухни > общей,2,0.000
0,Цена <= 0,0,0.000
1,Общая площадь <= 0,0,0.000
2,Жилая площадь < 0,0,0.000
4,Жилая площадь > общей,0,0.000
3,Площадь кухни < 0,0,0.000
7,Количество комнат < 0,0,0.000
8,Количество комнат = 0,0,0.000
9,Этаж <= 0,0,0.000


Бесконечные значения по колонкам:


price             0
total_area        0
living_area       0
kitchen_area      0
rooms             0
floor             0
floors_total      0
ceiling_height    0
build_year        0
flats_count       0
latitude          0
longitude         0
dtype: int64

In [10]:
#Самые низкие и высокие цены
#Цена квадратного метра здесь нужна только для диагностики.
#В признаки модели её включать нельзя: 
#она вычисляется через целевую переменную price.

price_diagnostics = df.assign(
    price_per_m2=df["price"] / df["total_area"]
)

price_columns = [
    "flat_id",
    "building_id",
    "price",
    "total_area",
    "price_per_m2",
    "rooms",
    "is_apartment",
]

print("10 квартир с минимальной ценой:")
display(
    price_diagnostics.nsmallest(10, "price")[price_columns]
)

print("5 квартир с максимальной ценой:")
display(
    price_diagnostics.nlargest(5, "price")[price_columns]
)

print("10 квартир с минимальной ценой за квадратный метр:")
display(
    price_diagnostics.nsmallest(10, "price_per_m2")[price_columns]
)

10 квартир с минимальной ценой:


,flat_id,building_id,price,total_area,price_per_m2,rooms,is_apartment
69633,69633,20672,11,42.000,0.262,2,False
107198,107198,9797,19,70.000,0.271,2,False
104573,104573,5381,85,59.000,1.441,1,False
60417,60417,22183,1500,36.000,41.667,1,False
61931,61931,3393,2000,30.000,66.667,1,False
76800,76800,6439,2000,38.000,52.632,1,False
128961,128961,20428,2000,40.000,50.000,1,False
126489,126489,12495,2300,38.200,60.209,1,False
137162,137162,6439,2400,40.000,60.000,1,False
59840,59840,14009,2500,36.000,69.444,1,True


5 квартир с максимальной ценой:


,flat_id,building_id,price,total_area,price_per_m2,rooms,is_apartment
85787,85787,19982,9873737728,220.000,"44,880,626.036",5,False
83773,83773,6108,9799999488,37.000,"264,864,851.027",2,False
40855,40855,10733,9200000000,45.000,"204,444,444.444",2,False
118083,118083,22134,8147034112,208.000,"39,168,433.231",4,False
50455,50455,22418,4686425088,103.000,"45,499,272.699",3,False


10 квартир с минимальной ценой за квадратный метр:


,flat_id,building_id,price,total_area,price_per_m2,rooms,is_apartment
69633,69633,20672,11,42.000,0.262,2,False
107198,107198,9797,19,70.000,0.271,2,False
104573,104573,5381,85,59.000,1.441,1,False
60417,60417,22183,1500,36.000,41.667,1,False
135997,135997,22158,4000,95.000,42.105,2,True
21821,21821,6386,3000,60.000,50.000,2,False
128961,128961,20428,2000,40.000,50.000,1,False
76800,76800,6439,2000,38.000,52.632,1,False
130229,130229,24004,3000,54.000,55.556,2,False
90369,90369,1243,3890,70.000,55.571,2,False


In [11]:
#Нулевые площади и признак студии
zero_area_report = pd.Series({
    "Жилая площадь = 0": int(df["living_area"].eq(0).sum()),
    "Площадь кухни = 0": int(df["kitchen_area"].eq(0).sum()),
    "Обе площади = 0": int(
        (df["living_area"].eq(0) & df["kitchen_area"].eq(0)).sum()
    ),
})

display(zero_area_report)

print("Значения studio:")
display(df["studio"].value_counts(dropna=False))

print("Нулевые площади по типу объекта:")
display(
    df.assign(
        living_zero=df["living_area"].eq(0),
        kitchen_zero=df["kitchen_area"].eq(0),
    )
    .groupby(["studio", "is_apartment"], dropna=False)
    .agg(
        rows_count=("flat_id", "size"),
        living_zero_count=("living_zero", "sum"),
        kitchen_zero_count=("kitchen_zero", "sum"),
    )
)

Жилая площадь = 0    18588
Площадь кухни = 0    11701
Обе площади = 0       8449
dtype: int64

Значения studio:


studio
False    141362
Name: count, dtype: int64

Нулевые площади по типу объекта:


rows_count  living_zero_count  kitchen_zero_count
studio is_apartment                                                   
False  False             139990              18226               11425
       True                1372                362                 276

In [12]:
#Большая высота потолков
#Порог 5 м здесь — условие для просмотра, а не готовая граница удаления.

high_ceiling = df.loc[df["ceiling_height"] > 5]

print("Количество квартир с потолками выше 5 м:", len(high_ceiling))
print("Количество соответствующих домов:",
      high_ceiling["building_id"].nunique())

print("Высота потолков и количество домов:")
display(
    high_ceiling[["building_id", "ceiling_height"]]
    .drop_duplicates()
    ["ceiling_height"]
    .value_counts()
    .sort_index()
)

display(
    high_ceiling[
        ["building_id", "ceiling_height", "build_year", "floors_total"]
    ]
    .drop_duplicates()
    .sort_values("ceiling_height", ascending=False)
    .head(20)
)

Количество квартир с потолками выше 5 м: 29
Количество соответствующих домов: 10
Высота потолков и количество домов:


ceiling_height
5.200     1
6.000     5
7.000     2
8.000     1
27.000    1
Name: count, dtype: int64

,building_id,ceiling_height,build_year,floors_total
83673,3852,27.000,1961,5
7486,12052,8.000,1977,5
50624,632,7.000,1931,6
15762,631,7.000,1931,5
8689,249,6.000,1914,7
18581,184,6.000,1912,7
107982,250,6.000,1914,24
68033,183,6.000,1912,6
116881,248,6.000,1914,6
56286,282,5.200,1917,5


In [13]:
#Насколько сильно площади противоречат друг другу
area_diagnostics = df.assign(
    area_excess=(
        df["living_area"]
        + df["kitchen_area"]
        - df["total_area"]
    )
)

print("Количество строк при разных величинах превышения:")
display(pd.Series({
    "Больше 0 м²": int(area_diagnostics["area_excess"].gt(0).sum()),
    "Больше 0.01 м²": int(area_diagnostics["area_excess"].gt(0.01).sum()),
    "Больше 0.1 м²": int(area_diagnostics["area_excess"].gt(0.1).sum()),
    "Больше 1 м²": int(area_diagnostics["area_excess"].gt(1).sum()),
}))

print("Наибольшие превышения:")
display(
    area_diagnostics.loc[area_diagnostics["area_excess"] > 0]
    .nlargest(15, "area_excess")[
        [
            "flat_id",
            "total_area",
            "living_area",
            "kitchen_area",
            "area_excess",
        ]
    ]
)

print("Кухня больше общей площади:")
display(
    df.loc[
        df["kitchen_area"] > df["total_area"],
        ["flat_id", "total_area", "living_area", "kitchen_area"],
    ]
)

Количество строк при разных величинах превышения:


Больше 0 м²       152
Больше 0.01 м²    127
Больше 0.1 м²     127
Больше 1 м²       113
dtype: int64

Наибольшие превышения:


,flat_id,total_area,living_area,kitchen_area,area_excess
114453,114453,39.000,20.000,102.000,83.000
114197,114197,59.000,43.000,65.000,49.000
55600,55600,162.400,145.000,45.000,27.600
29747,29747,73.000,62.000,32.000,21.000
55583,55583,73.000,62.000,32.000,21.000
128169,128169,73.000,62.000,32.000,21.000
69828,69828,51.300,49.000,20.000,17.700
14055,14055,154.600,152.300,18.800,16.500
138545,138545,103.800,80.000,40.000,16.200
102289,102289,46.000,38.000,24.000,16.000


Кухня больше общей площади:


,flat_id,total_area,living_area,kitchen_area
114197,114197,59.000,43.000,65.000
114453,114453,39.000,20.000,102.000


In [ ]:
%%writefile {project_dir}/dags/real_estate_cleaning.py

#Таким образом, функции останутся видны в ноутбуке,
#а DAG сможет импортировать этот же код.

import numpy as np
import pandas as pd


DATASET_COLUMNS = [
    "flat_id", "building_id", "floor", "is_apartment", "kitchen_area",
    "living_area", "rooms", "studio", "total_area", "price", "build_year",
    "building_type_int", "latitude", "longitude", "ceiling_height",
    "flats_count", "floors_total", "has_elevator",
]


def remove_exact_duplicates(frame):
    """Удаляем только полностью одинаковые строки, включая flat_id."""
    duplicate_mask = frame.duplicated(keep="first")

    rejected = frame.loc[duplicate_mask].copy()
    rejected["reject_reason"] = "exact_duplicate"

    result = frame.loc[~duplicate_mask].copy()

    if result["flat_id"].isna().any():
        raise ValueError("Обнаружен пустой flat_id.")

    if result["flat_id"].duplicated().any():
        raise ValueError("Один flat_id имеет разные характеристики.")

    return result, rejected


def clean_feature_values(frame, area_tolerance=0.1, max_ceiling_height=10.0):
    """Обозначаем неизвестные и противоречивые значения пропусками."""
    result = frame.copy()
    report = {}

    for column in ["living_area", "kitchen_area"]:
        values = result[column].astype("float64")

        invalid = values.notna() & (
            ~np.isfinite(values) | values.le(0)
        )

        report[f"{column}_invalid_to_nan"] = int(invalid.sum())
        result[column] = values.mask(invalid)

        # Сначала исправляем противоречия отдельных площадей.
        too_large = (
            result[column] > result["total_area"] + area_tolerance
        )

        report[f"{column}_exceeds_total"] = int(too_large.sum())
        result.loc[too_large, column] = np.nan

    # Затем проверяем сумму оставшихся известных площадей.
    excess = (
        result["living_area"]
        + result["kitchen_area"]
        - result["total_area"]
    )

    inconsistent = excess > area_tolerance
    report["area_sum_conflicts"] = int(inconsistent.sum())

    result.loc[
        inconsistent,
        ["living_area", "kitchen_area"],
    ] = np.nan

    ceiling = result["ceiling_height"].astype("float64")

    invalid_ceiling = ceiling.notna() & (
        ~np.isfinite(ceiling)
        | ceiling.le(0)
        | ceiling.gt(max_ceiling_height)
    )

    report["ceiling_height_invalid_to_nan"] = int(
        invalid_ceiling.sum()
    )
    result["ceiling_height"] = ceiling.mask(invalid_ceiling)

    return result, report


def clean_dataset(
    frame,
    min_price_per_m2=1000.0,
    max_price_per_m2=10_000_000.0,
    area_tolerance=0.1,
    max_ceiling_height=10.0,
):
    """Возвращает очищенные данные, исключённые строки и отчёт."""
    if not 0 < min_price_per_m2 < max_price_per_m2:
        raise ValueError(
            "Некорректные границы цены за квадратный метр."
        )

    if area_tolerance < 0 or max_ceiling_height <= 0:
        raise ValueError("Некорректные параметры очистки признаков.")

    missing = set(DATASET_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"Отсутствуют колонки: {sorted(missing)}")

    # Выбираем только исходные колонки: диагностические не попадут в результат.
    original = (
        frame[DATASET_COLUMNS]
        .copy()
        .reset_index(drop=True)
    )

    work, exact_duplicates = remove_exact_duplicates(original)

    total = work["total_area"].astype("float64")
    price = work["price"].astype("float64")

    total_ok = np.isfinite(total) & total.gt(0)
    price_ok = np.isfinite(price) & price.gt(0)

    # Используется только для контроля качества, не как признак модели.
    price_per_m2 = price / total.where(total_ok)

    reasons = pd.Series("", index=work.index, dtype="object")

    reasons.loc[~total_ok] = "invalid_total_area"
    reasons.loc[total_ok & ~price_ok] = "invalid_price"

    reasons.loc[
        total_ok & price_ok & price_per_m2.lt(min_price_per_m2)
    ] = "price_per_m2_too_low"

    reasons.loc[
        total_ok & price_ok & price_per_m2.gt(max_price_per_m2)
    ] = "price_per_m2_too_high"

    excluded = work.loc[reasons.ne("")].copy()
    excluded["reject_reason"] = reasons.loc[reasons.ne("")]

    retained = work.loc[reasons.eq("")].copy()

    cleaned, feature_report = clean_feature_values(
        retained,
        area_tolerance=area_tolerance,
        max_ceiling_height=max_ceiling_height,
    )

    rejected = pd.concat(
        [exact_duplicates, excluded],
        ignore_index=True,
    )
    cleaned = cleaned.reset_index(drop=True)

    report = {
        "input_rows": int(len(original)),
        "exact_duplicates_removed": int(len(exact_duplicates)),
        "invalid_total_area_rows": int(
            reasons.eq("invalid_total_area").sum()
        ),
        "invalid_price_rows": int(
            reasons.eq("invalid_price").sum()
        ),
        "price_per_m2_too_low_rows": int(
            reasons.eq("price_per_m2_too_low").sum()
        ),
        "price_per_m2_too_high_rows": int(
            reasons.eq("price_per_m2_too_high").sum()
        ),
        **feature_report,
        "rejected_rows": int(len(rejected)),
        "output_rows": int(len(cleaned)),
    }

    if len(cleaned) + len(rejected) != len(original):
        raise RuntimeError("Нарушен баланс строк при очистке.")

    return cleaned, rejected, report

Writing /home/mle-user/mle_project/mle-project_Mandrygina_Olga_M25-555/dags/real_estate_cleaning.py


In [15]:
#Очистка

import importlib
import sys

dags_path = str(project_dir / "dags")
if dags_path not in sys.path:
    sys.path.insert(0, dags_path)

import real_estate_cleaning
importlib.reload(real_estate_cleaning)

cleaning_params = {
    "min_price_per_m2": 1000.0,
    "max_price_per_m2": 10_000_000.0,
    "area_tolerance": 0.1,
    "max_ceiling_height": 10.0,
}

clean_df, rejected_df, cleaning_report = (
    real_estate_cleaning.clean_dataset(
        df,
        **cleaning_params,
    )
)

display(pd.Series(cleaning_report, name="Количество"))

input_rows                       141362
exact_duplicates_removed              0
invalid_total_area_rows               0
invalid_price_rows                    0
price_per_m2_too_low_rows           132
price_per_m2_too_high_rows           15
living_area_invalid_to_nan        18557
living_area_exceeds_total             0
kitchen_area_invalid_to_nan       11664
kitchen_area_exceeds_total            2
area_sum_conflicts                  122
ceiling_height_invalid_to_nan         1
rejected_rows                       147
output_rows                      141215
Name: Количество, dtype: int64

In [16]:
#Посмотри причины исключения и пропуски после очистки

print("Причины исключения строк:")
display(
    rejected_df["reject_reason"]
    .value_counts()
    .rename_axis("Причина")
    .to_frame("Количество")
)

print("Пропуски после очистки:")
missing_after = pd.DataFrame({
    "Количество": clean_df.isna().sum(),
    "Доля, %": clean_df.isna().mean().mul(100).round(2),
})

display(
    missing_after.loc[missing_after["Количество"] > 0]
    .sort_values("Количество", ascending=False)
)

print(f"До очистки:     {len(df):,}")
print(f"После очистки:  {len(clean_df):,}")
print(f"Исключено:      {len(rejected_df):,}")
print(f"Сохранено:      {len(clean_df) / len(df):.2%}")


Причины исключения строк:


,Количество
Причина,
price_per_m2_too_low,132
price_per_m2_too_high,15


Пропуски после очистки:


,Количество,"Доля, %"
living_area,18679,13.230
kitchen_area,11788,8.350
ceiling_height,1,0.000


До очистки:     141,362
После очистки:  141,215
Исключено:      147
Сохранено:      99.90%
